# Notebook 2 — Illustration Cropping with YOLOv8

This notebook uses my trained YOLOv8 model for detecting the categories "Illustration" and "Editorial Cartoon" in the periodical, cropping them out and saving them into their own sub-folders.

I chose these categories as they are the ones relevant for finding the recurring characters in the "Die Bombe". The create crops will then be vector embedded and used for clustering in the next step.

## Metadata
This cropping-process also created a new set of metadata based on the metadata created when downloading the pages. Each created crops gets a unique identifier (column "crop_ID") and gets linked with the page_ID correpsonding to the page that depicts it. As there is a great possibility of multiple editorial cartoons/illustrations occuring on the same page, the unique Page-ID enables the notebook to assign several crops to the same page without problems.

## Uniqueness of Crops
As the trained YOLOv8 model sometimes assigns two different lables to the same depiction, an overlap-reckoning is used to ensure the label with the higher predicition-rate is saved and no crops of the same image exists two times (e. g. once as an illustration and once as an editorial cartoon).

This notebook:
- loads a fine-tuned YOLOv8 model
- runs inference on downloaded page JPGs
- saves crops into category folders (`Crops/Editorial-Cartoon`, `Crops/Illustration`)
- writes one metadata row per crop **immediately** (safe on interruption)
- supports resume via `done_images.txt` and `state.json`


## 1) Installation & Imports

In [ ]:
# === Install (Colab) ===
!pip -q install ultralytics pandas requests tqdm opencv-python

# === Imports ===
import os, re, csv, json, time
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import cv2
from ultralytics import YOLO


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 2) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


## 3) Configurate paths & parameters
This cell is used to set the paths for the cropping process.
In this cell the inference settings can be chosen as well as the categories which should be considered by the model for the cropping run.

In [ ]:
# =========================
# CONFIG (EDIT THESE PATHS)
# =========================

# Page metadata CSV from page-downloader notebook (Notebook 0)
# It should include: page_id, iiif_jpg_url, local_jpg_path
PAGES_METADATA_CSV = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata.csv"

# Folder containing pages (fallback if CSV is missing/incomplete)
PAGES_JPG_ROOT = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/pages_jpg"

# Fine-tuned YOLO weights
WEIGHTS_PATH = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Models/runs/yolov8m_final-pipeline_run-001/weights/best.pt"

# Output root for crops + metadata
OUTPUT_BASE_DIR = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Crops_Bombe_final"

# Inference settings
IMG_SIZE = 1280
CONF_THRESHOLD = 0.25

# Which categories to KEEP (directories will be created from these)
KEEP_CATEGORIES = {"Editorial Cartoon", "Illustration"}  # set to None to keep all classes

# Optional: restrict maximum pages for quick test (None = all)
MAX_PAGES = None  # e.g. 50 for testing

print("CONFIG OK")

CONFIG OK


## 4) Auxiliary functions
In this cell the auxiliary functions are defined.
They concern logging bad images (jpg-files that could not be processed) to make it possible to look them up, creating the metadata-files for the crops and creating IIIF-linkage for the crops in the metadata for enhancing interoperability. It also contains functions for ensuring resuming the cropping process after an interuption by creating a log for the images that were already processed.

In [ ]:
# =========================
# Auxiliary Functions
# =========================

CROPS_ROOT = Path(OUTPUT_BASE_DIR)
CROPS_ROOT.mkdir(parents=True, exist_ok=True)

META_DIR = CROPS_ROOT / "metadata"
META_DIR.mkdir(parents=True, exist_ok=True)

CROPS_METADATA_CSV = META_DIR / "Crops_Bombe_metadata.csv"
DONE_IMAGES_TXT    = META_DIR / "done_images.txt"
STATE_JSON         = META_DIR / "state.json"

# =========================
# BAD IMAGE LOG
# =========================

BAD_IMAGES_CSV = META_DIR / "bad_images.csv"

BAD_IMAGE_HEADER = [
    "img_path",
    "page_id",
    "reason",
    "file_size_bytes",
]

def log_bad_image(img_path, page_id, reason):
    try:
        size = Path(img_path).stat().st_size
    except Exception:
        size = None

    append_csv_rows(
        BAD_IMAGES_CSV,
        [{
            "img_path": img_path,
            "page_id": page_id,
            "reason": reason,
            "file_size_bytes": size,
        }],
        BAD_IMAGE_HEADER
    )

# ---- Crop metadata schema ----
CROP_CSV_HEADER = [
    "crop_id",
    "page_id",
    "category",
    "category_number",
    "bbox_xyxy",          # "(x1,y1,x2,y2)"
    "bbox_xywh",          # "(x,y,w,h)"   -- better for IIIF region
    "confidence",
    "iiif_crop_url",
    "iiif_page_url",
    "local_crop_path",
    "model_weights",
    "imgsz",
    "conf_threshold",
]

def append_csv_rows(path: Path, rows: list[dict], header: list[str]):
    """Append rows to CSV; creates file with header if missing."""
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    file_exists = path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=header)
        if not file_exists:
            w.writeheader()
        for r in rows:
            w.writerow(r)

def load_done_images(done_path: Path) -> set[str]:
    if not done_path.exists():
        return set()
    with open(done_path, "r", encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip()}

def mark_image_done(done_path: Path, img_path: str):
    with open(done_path, "a", encoding="utf-8") as f:
        f.write(img_path + "\n")

def load_crop_counter(state_path: Path) -> int:
    if not state_path.exists():
        return 0
    try:
        return int(json.loads(state_path.read_text(encoding="utf-8")).get("last_crop_index", 0))
    except Exception:
        return 0

def save_crop_counter(state_path: Path, last_idx: int):
    state_path.write_text(json.dumps({"last_crop_index": int(last_idx)}), encoding="utf-8")

def make_crop_id(counter: int) -> str:
    return f"bom-crop_{counter:08d}"

def normalize_category_for_folder(cat: str) -> str:
    """Folder-friendly category name."""
    return str(cat).strip().replace(" ", "-")

def page_id_from_image_path(img_path: str) -> str:
    """
    Preferred: page filename stem, e.g. bom18710108_001.jpg -> bom18710108_001
    If the stem contains more stuff, try to extract the bom... pattern.
    """
    stem = Path(img_path).stem
    m = re.search(r"(bom\d{8}_\d{3})", stem, flags=re.IGNORECASE)
    return m.group(1) if m else stem

def iiif_crop_from_page_jpg_url(iiif_page_jpg_url: str, x: int, y: int, w: int, h: int) -> str | None:
    """
    Convert IIIF page URL like:
      .../full/max/0/default.jpg
    into crop URL:
      .../{x},{y},{w},{h}/full/0/default.jpg
    """
    if not iiif_page_jpg_url:
        return None

    region = f"{x},{y},{w},{h}"
    out = re.sub(
        r"/full/[^/]+/0/default\.(jpg|jpeg|png)$",
        f"/{region}/full/0/default.jpg",
        iiif_page_jpg_url,
        flags=re.IGNORECASE
    )
    return out if out != iiif_page_jpg_url else None

print("Auxiliary functions ready.")


Auxiliary functions ready.


## 5) Load page metadata mapping and build image list

In [ ]:
# =========================
# PAGE METADATA MAPPING
# =========================
page2iiif = {}
page2local = {}

if PAGES_METADATA_CSV and os.path.exists(PAGES_METADATA_CSV):
    df_pages = pd.read_csv(
        PAGES_METADATA_CSV,
        usecols=["page_id", "iiif_jpg_url", "local_jpg_path"],
        dtype="string",
        engine="pyarrow",  # if this errors in Colab, remove this line
    )

    df_pages["page_id"] = df_pages["page_id"].str.strip()
    df_pages = df_pages[df_pages["page_id"].notna() & (df_pages["page_id"] != "")]

    df_pages["iiif_jpg_url"] = df_pages["iiif_jpg_url"].fillna("").str.strip()
    df_pages["local_jpg_path"] = df_pages["local_jpg_path"].fillna("").str.strip()

    page2iiif = dict(
        zip(
            df_pages["page_id"].tolist(),
            df_pages["iiif_jpg_url"].replace({"": None}).tolist(),
        )
    )
    page2local = dict(
        zip(
            df_pages["page_id"].tolist(),
            df_pages["local_jpg_path"].replace({"": None}).tolist(),
        )
    )

    print(f"Loaded {len(page2iiif)} page→IIIF mappings from CSV.")
else:
    print("[WARN] PAGES_METADATA_CSV not found. IIIF crop links will be None.")

# Resume
done_images = set(load_done_images(DONE_IMAGES_TXT))
crop_counter = load_crop_counter(STATE_JSON)

# Build image list
if page2local:
    image_paths = [
        p for p in page2local.values()
        if isinstance(p, str)
        and p
        and p.lower().endswith((".jpg", ".jpeg"))
        and p not in done_images   # IMPORTANT: compare path to path
    ]
    print(f"Using local_jpg_path from CSV: {len(image_paths)} images (not yet processed).")
else:
    image_paths = [str(p) for p in Path(PAGES_JPG_ROOT).rglob("*.jpg")]
    # If you want resume behavior even in scan mode:
    image_paths = [p for p in image_paths if p not in done_images]
    print(f"Scanning folder: {len(image_paths)} images (not yet processed).")

# Optional quick limit
if MAX_PAGES is not None:
    image_paths = image_paths[: int(MAX_PAGES)]
    print(f"MAX_PAGES applied -> {len(image_paths)} images.")

print(f"Resume status: {len(done_images)} pages already processed.")
print(f"Next crop index: {crop_counter + 1}")

Loaded 21103 page→IIIF mappings from CSV.
Using local_jpg_path from CSV: 1435 images (not yet processed).
Resume status: 19668 pages already processed.
Next crop index: 30170


## 6) Load the fine-tuned YOLO model

In [ ]:
# =========================
# MODEL LOAD
# =========================

model = YOLO(WEIGHTS_PATH)
print("Loaded weights:", WEIGHTS_PATH)
print("Model classes:", model.names)

Loaded weights: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Models/runs/yolov8m_final-pipeline_run-001/weights/best.pt
Model classes: {0: 'Advertisement', 1: 'Comic', 2: 'Editorial Cartoon', 3: 'Headline', 4: 'Illustration', 5: 'Map', 6: 'Photograph'}


## 7) Run detection
This will:
- create category folders
- write **one row per crop immediately** for safe resuming after an interruption
- record processed pages in `done_images.txt`
- update crop counter in `state.json`
- ensure that if crops have an IOU of at least 0.7, only the crop with the higher prediciton certainty will be saved

In [ ]:
# =========================
# MAIN LOOP
# =========================

# --- IoU enforcement setting ---
IOU_SUPPRESS_THRESHOLD = 0.70  # "big overlap" threshold; tune if needed (e.g., 0.6–0.85)

def _iou_xyxy(a, b):
    """IoU for boxes in [x1,y1,x2,y2] format."""
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return (inter / union) if union > 0 else 0.0

def suppress_by_iou_keep_best(dets, iou_thr):
    """
    dets: list of dicts with keys: xyxy, conf, cls_id, cls_name
    Sort by confidence desc, then greedily keep detections that don't overlap too much.
    """
    dets = sorted(dets, key=lambda d: d["conf"], reverse=True)
    kept = []
    for d in dets:
        if all(_iou_xyxy(d["xyxy"], k["xyxy"]) < iou_thr for k in kept):
            kept.append(d)
    return kept

# Prepare allowed categories set
allowed = None
if KEEP_CATEGORIES is not None:
    allowed = {str(x).strip() for x in KEEP_CATEGORIES}

# Corruption heuristic: conservative safeguard
MIN_BYTES = 50_000  # 50 KB

processed_now = 0
total_crops_now = 0

for img_path in tqdm(image_paths, desc="Detect+Crop", total=len(image_paths)):
    img_path = str(img_path)

    if img_path in done_images:
        continue

    # page_id early (used for logging)
    page_id = page_id_from_image_path(img_path)

    # ---- SAFE SKIP 1: missing / too small ----
    try:
        size = Path(img_path).stat().st_size
    except FileNotFoundError:
        print(f"[SKIP] file missing: {img_path}")
        log_bad_image(img_path=img_path, page_id=page_id, reason="file_missing")
        mark_image_done(DONE_IMAGES_TXT, img_path)
        done_images.add(img_path)
        processed_now += 1
        continue

    if size < MIN_BYTES:
        print(f"[SKIP] file too small (<{MIN_BYTES} bytes): {img_path} ({size} bytes)")
        log_bad_image(img_path=img_path, page_id=page_id, reason="file_too_small")
        mark_image_done(DONE_IMAGES_TXT, img_path)
        done_images.add(img_path)
        processed_now += 1
        continue

    # ---- SAFE SKIP 2: unreadable by OpenCV ----
    img = cv2.imread(img_path)
    if img is None:
        print(f"[SKIP] unreadable/corrupt image: {img_path}")
        log_bad_image(img_path=img_path, page_id=page_id, reason="cv2_imread_failed")
        mark_image_done(DONE_IMAGES_TXT, img_path)
        done_images.add(img_path)
        processed_now += 1
        continue

    h, w = img.shape[:2]
    iiif_page_url = page2iiif.get(page_id) if page2iiif else None

    # ---- YOLO inference ----
    results = model(img, imgsz=IMG_SIZE, conf=CONF_THRESHOLD, verbose=False)
    if not results:
        mark_image_done(DONE_IMAGES_TXT, img_path)
        done_images.add(img_path)
        processed_now += 1
        continue

    res = results[0]
    boxes = res.boxes
    if boxes is None or len(boxes) == 0:
        mark_image_done(DONE_IMAGES_TXT, img_path)
        done_images.add(img_path)
        processed_now += 1
        continue

    # ---- Build detection list ----
    dets = []
    for box in boxes:
        cls_id = int(box.cls[0].item())
        conf   = float(box.conf[0].item())
        cls_name = model.names[cls_id]

        if allowed is not None and cls_name not in allowed:
            continue

        # Clamp bbox
        x1_f, y1_f, x2_f, y2_f = box.xyxy[0].tolist()
        x1 = max(0, min(w - 1, int(x1_f)))
        y1 = max(0, min(h - 1, int(y1_f)))
        x2 = max(0, min(w,     int(x2_f)))
        y2 = max(0, min(h,     int(y2_f)))

        if x2 <= x1 or y2 <= y1:
            continue

        dets.append({
            "xyxy": [x1, y1, x2, y2],
            "conf": conf,
            "cls_id": cls_id,
            "cls_name": cls_name
        })

    # ---- IoU suppression (keep best confidence if overlap is big) ----
    kept_dets = suppress_by_iou_keep_best(dets, IOU_SUPPRESS_THRESHOLD)

    rows = []

    # ---- Save crops only for kept detections ----
    for d in kept_dets:
        cls_id = d["cls_id"]
        conf   = d["conf"]
        cls_name = d["cls_name"]
        x1, y1, x2, y2 = d["xyxy"]

        crop_img = img[y1:y2, x1:x2]

        # Crop ID
        crop_counter += 1
        crop_id = make_crop_id(crop_counter)

        # Folder per category
        folder_name = normalize_category_for_folder(cls_name)
        out_dir = CROPS_ROOT / folder_name
        out_dir.mkdir(parents=True, exist_ok=True)

        crop_path = out_dir / f"{crop_id}.jpg"
        cv2.imwrite(str(crop_path), crop_img)

        # IIIF crop url
        crop_w = x2 - x1
        crop_h = y2 - y1
        iiif_crop_url = iiif_crop_from_page_jpg_url(iiif_page_url, x1, y1, crop_w, crop_h)

        rows.append({
            "crop_id": crop_id,
            "page_id": page_id,
            "category": folder_name,
            "category_number": cls_id,
            "bbox_xyxy": f"({x1},{y1},{x2},{y2})",
            "bbox_xywh": f"({x1},{y1},{crop_w},{crop_h})",
            "confidence": conf,
            "iiif_crop_url": iiif_crop_url,
            "iiif_page_url": iiif_page_url,
            "local_crop_path": str(crop_path),
            "model_weights": WEIGHTS_PATH,
            "imgsz": IMG_SIZE,
            "conf_threshold": CONF_THRESHOLD,
        })

        # Update counter frequently
        save_crop_counter(STATE_JSON, crop_counter)

    # Write rows for this page immediately
    append_csv_rows(CROPS_METADATA_CSV, rows, CROP_CSV_HEADER)
    total_crops_now += len(rows)

    # Mark page done
    mark_image_done(DONE_IMAGES_TXT, img_path)
    done_images.add(img_path)
    processed_now += 1

print("\nDone (or safely resumable).")
print("Processed pages this run:", processed_now)
print("Crops written this run:", total_crops_now)
print("Crop metadata CSV:", CROPS_METADATA_CSV)
print("Bad images CSV:", BAD_IMAGES_CSV)
print("State:", STATE_JSON)

Detect+Crop: 100%|██████████| 1435/1435 [1:08:40<00:00,  2.87s/it]


Done (or safely resumable).
Processed pages this run: 1435
Crops written this run: 1097
Crop metadata CSV: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Crops_Bombe_final/metadata/Crops_Bombe_metadata.csv
Bad images CSV: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Crops_Bombe_final/metadata/bad_images.csv
State: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Crops_Bombe_final/metadata/state.json


## 8) Quick check — number of crops per category (optional)

In [ ]:
from pathlib import Path
import pandas as pd

if Path(CROPS_METADATA_CSV).exists():
    df = pd.read_csv(CROPS_METADATA_CSV)
    print("Rows:", len(df))
    print(df["category"].value_counts().head(20))
else:
    print("No crop metadata CSV found yet:", CROPS_METADATA_CSV)


Rows: 31266
category
Illustration         27469
Editorial-Cartoon     3797
Name: count, dtype: int64
